# Input Parser (Hebrew) — Single Step Evaluation

Evaluates `input_parser_node` on **Hebrew input** across 5 dimensions:
1. **Action Classification** — correct routing decision (deterministic)
2. **Item Count** — correct number of food items extracted (deterministic)
3. **Amount Accuracy** — gram conversion within ±20% tolerance (deterministic)
4. **Date Parsing** — correct date/time extraction (deterministic)
5. **Food Name Quality** — search-friendly normalization (LLM-as-judge)

In [14]:
import sys
import os
from pathlib import Path

# Add project root to path (notebooks/evals/ -> project root)
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, ".env"))

from langsmith import Client
from langchain_core.messages import HumanMessage

from src.agents.nodes.input_node import input_parser_node

client = Client()
print("Setup complete")

Setup complete


## Dataset: Input Parser Hebrew

17 examples in Hebrew covering all 4 action types with full reference outputs.

In [15]:
examples = [

    # --- LOG_FOOD: Basic single item ---

    {

        "question": "אכלתי 200 גרם עוף",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Chicken", "amount": 200.0}],

        "item_count": 1,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: Multi-item ---

    {

        "question": "תרשום בננה ו-100 גרם אורז",

        "action": "LOG_FOOD",

        "items": [

            {"food_name": "Banana", "amount": 120.0},

            {"food_name": "Rice", "amount": 100.0}

        ],

        "item_count": 2,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: No verb, just food + quantity ---

    {

        "question": "200 גרם חזה עוף",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Chicken Breast", "amount": 200.0}],

        "item_count": 1,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: Single word, no quantity (default serving) ---

    {

        "question": "קפה",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Coffee", "amount": 240.0}],

        "item_count": 1,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: Meal decomposition ---

    {

        "question": "פסטה עם גבינה לצהריים",

        "action": "LOG_FOOD",

        "items": [

            {"food_name": "Pasta", "amount": 200.0},

            {"food_name": "Cheese", "amount": 30.0}

        ],

        "item_count": 2,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: Unit conversion (cups -> grams) ---

    {

        "question": "אכלתי כוס אורז",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Rice", "amount": 158.0}],

        "item_count": 1,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: Unit conversion (slices -> grams) ---

    {

        "question": "2 פרוסות לחם",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Bread", "amount": 60.0}],

        "item_count": 1,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: "חלבון" in text should NOT confuse with stats ---

    {

        "question": "שתיתי שייק חלבון אחרי אימון",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Protein Shake", "amount": 300.0}],

        "item_count": 1,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: With relative time ---

    {

        "question": "אכלתי 200 גרם עוף לפני שעתיים",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Chicken", "amount": 200.0}],

        "item_count": 1,

        "consumed_at": "RELATIVE",

        "start_date": None,

        "end_date": None,

    },

    # --- LOG_FOOD: With specific date ---

    {

        "question": "אכלתי בננה אתמול",

        "action": "LOG_FOOD",

        "items": [{"food_name": "Banana", "amount": 120.0}],

        "item_count": 1,

        "consumed_at": "YESTERDAY_NOON",

        "start_date": None,

        "end_date": None,

    },

    # --- QUERY_FOOD_INFO ---

    {

        "question": "כמה חלבון יש בביצה?",

        "action": "QUERY_FOOD_INFO",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- QUERY_FOOD_INFO: Could confuse with LOG_FOOD ---

    {

        "question": "כמה קלוריות יש בבננה?",

        "action": "QUERY_FOOD_INFO",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- QUERY_DAILY_STATS: Basic ---

    {

        "question": "מה אכלתי היום?",

        "action": "QUERY_DAILY_STATS",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- QUERY_DAILY_STATS: With date range (3 days) ---

    {

        "question": "סטטיסטיקות של 3 ימים אחרונים",

        "action": "QUERY_DAILY_STATS",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": "RELATIVE_3_DAYS_AGO",

        "end_date": "TODAY",

    },

    # --- CHITCHAT ---

    {

        "question": "היי מה נשמע?",

        "action": "CHITCHAT",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": None,

        "end_date": None,

    },

    # --- QUERY_DAILY_STATS: Weekly range ---

    {

        "question": "מה אכלתי בשבוע האחרון",

        "action": "QUERY_DAILY_STATS",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": "RELATIVE_7_DAYS_AGO",

        "end_date": "TODAY",

    },

    # --- QUERY_DAILY_STATS: Specific macro question over range ---

    {

        "question": "כמה גרם חלבון אכלתי בממוצע בשבוע האחרון",

        "action": "QUERY_DAILY_STATS",

        "items": [],

        "item_count": 0,

        "consumed_at": None,

        "start_date": "RELATIVE_7_DAYS_AGO",

        "end_date": "TODAY",

    },

]



# Dataset created in LangSmith UI under fit-pal-agent project

dataset_id = "175cb9ae-e063-466c-a79a-c71db1d94ca2"

dataset_name = "Input Parser Hebrew"



# Upload examples if dataset is empty

existing = list(client.list_examples(dataset_id=dataset_id))

if not existing:

    client.create_examples(

        inputs=[{"question": ex["question"]} for ex in examples],

        outputs=[{k: v for k, v in ex.items() if k != "question"} for ex in examples],

        dataset_id=dataset_id,

    )

    print(f"Uploaded {len(examples)} examples to '{dataset_name}'")

else:

    print(f"Dataset '{dataset_name}' already has {len(existing)} examples")

Dataset 'Input Parser Hebrew' already has 17 examples


## Target Function

Calls `input_parser_node` directly with a minimal state dict.

In [16]:
async def run_input_parser(inputs: dict) -> dict:
    """Run input_parser_node and return structured outputs for evaluation."""
    state = {"messages": [HumanMessage(content=inputs["question"])]}
    result = await input_parser_node(state)
    return {
        "action": result["last_action"],
        "items": result["pending_food_items"],
        "item_count": len(result["pending_food_items"]),
        "consumed_at": str(result["consumed_at"]) if result.get("consumed_at") else None,
        "start_date": str(result["start_date"]) if result.get("start_date") else None,
        "end_date": str(result["end_date"]) if result.get("end_date") else None,
    }

In [17]:
# Smoke test — verify target function works with Hebrew input
test_result = await run_input_parser({"question": "אכלתי 200 גרם עוף"})
print(test_result)

2026-04-14 06:43:55 [info     ] Input parsed                   action=LOG_FOOD items=1
{'action': 'LOG_FOOD', 'items': [{'food_name': 'Chicken Breast', 'amount': 200.0, 'unit': 'g', 'original_text': '200 גרם עוף'}], 'item_count': 1, 'consumed_at': None, 'start_date': None, 'end_date': None}


## Evaluators

5 evaluators, each checking one dimension of the parser output.

In [18]:
def correct_action(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the parser selected the correct action/route."""
    return outputs["action"] == reference_outputs["action"]

In [19]:
def correct_item_count(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the parser extracted the correct number of food items."""
    return outputs["item_count"] == reference_outputs["item_count"]

In [20]:
def amount_accuracy(outputs: dict, reference_outputs: dict) -> dict:
    """Check if extracted amounts are within +/-20% of expected values.

    Returns a score between 0.0 and 1.0 (fraction of items within tolerance).
    Skips if no items expected (non-food actions).
    """
    expected_items = reference_outputs.get("items", [])
    actual_items = outputs.get("items", [])

    if not expected_items:
        return {"key": "amount_accuracy", "score": 1.0, "comment": "No items to check"}

    if len(actual_items) != len(expected_items):
        return {
            "key": "amount_accuracy",
            "score": 0.0,
            "comment": f"Item count mismatch: got {len(actual_items)}, expected {len(expected_items)}",
        }

    # Sort both lists by food_name for alignment
    expected_sorted = sorted(expected_items, key=lambda x: x["food_name"].lower())
    actual_sorted = sorted(actual_items, key=lambda x: x["food_name"].lower())

    within_tolerance = 0
    details = []
    for exp, act in zip(expected_sorted, actual_sorted):
        exp_amount = exp["amount"]
        act_amount = act["amount"]
        tolerance = exp_amount * 0.20
        is_close = abs(act_amount - exp_amount) <= tolerance
        if is_close:
            within_tolerance += 1
        details.append(
            f"{act.get('food_name', '?')}: {act_amount}g vs {exp_amount}g"
            f" {'(OK)' if is_close else '(FAIL)'}"
        )

    score = within_tolerance / len(expected_items)
    return {"key": "amount_accuracy", "score": score, "comment": "; ".join(details)}

In [21]:
from datetime import date, datetime, timedelta





def _resolve_date_sentinel(sentinel: str | None) -> str | None:

    """Convert sentinel values to actual date strings at eval time."""

    if sentinel is None:

        return None

    today = date.today()

    mapping = {

        "TODAY": str(today),
        "YESTERDAY": str(today - timedelta(days=1)),

        "YESTERDAY_NOON": str(

            datetime.combine(

                today - timedelta(days=1),

                datetime.min.replace(hour=12).time(),

            )

        ),

        # "last 3 days" inclusive of today = today - 2

        "RELATIVE_3_DAYS_AGO": str(today - timedelta(days=3)),  # excluding today

        # "last week" inclusive of today = today - 6

        "RELATIVE_7_DAYS_AGO": str(today - timedelta(days=7)),  # excluding today

    }

    if sentinel in mapping:

        return mapping[sentinel]

    if sentinel == "RELATIVE":

        return "RELATIVE"  # Special case: just check it's not None

    return sentinel





def _dates_equivalent(expected: str | None, actual: str | None) -> bool:

    """Compare date values, treating null and today as equivalent."""

    if expected is None and actual is None:

        return True

    if expected is not None and actual is not None:

        return expected[:10] == actual[:10]

    # One is None, one is not — accept if the non-None value is today

    today_str = str(date.today())

    if expected is None and actual is not None:

        return actual[:10] == today_str

    if expected is not None and actual is None:

        return expected[:10] == today_str

    return False





def correct_dates(outputs: dict, reference_outputs: dict) -> bool:

    """Check if date/time extraction matches expected values."""

    # --- consumed_at ---

    expected_consumed = _resolve_date_sentinel(reference_outputs.get("consumed_at"))

    actual_consumed = outputs.get("consumed_at")



    if expected_consumed == "RELATIVE":

        if actual_consumed is None:

            return False

    elif expected_consumed is not None:

        if actual_consumed is None:

            return False

        if expected_consumed[:10] != actual_consumed[:10]:

            return False

    else:

        if actual_consumed is not None:

            return False



    # --- start_date ---

    expected_start = _resolve_date_sentinel(reference_outputs.get("start_date"))

    actual_start = outputs.get("start_date")

    if not _dates_equivalent(expected_start, actual_start):

        return False



    # --- end_date ---

    expected_end = _resolve_date_sentinel(reference_outputs.get("end_date"))

    actual_end = outputs.get("end_date")

    if not _dates_equivalent(expected_end, actual_end):

        return False



    return True

In [22]:
from typing import Annotated

from langchain.chat_models import init_chat_model
from typing_extensions import TypedDict


class NameGrade(TypedDict):
    """Grade for food name normalization quality."""

    reasoning: Annotated[
        str, ..., "Step-by-step reasoning for the grade."
    ]
    is_acceptable: Annotated[
        bool, ..., "True if the name is a reasonable search-friendly normalization."
    ]


judge_instructions = """You are evaluating whether a food name has been properly normalized for database search.

The original user input is in Hebrew, but the extracted food name should be in English (for database lookup).

Rules:
- The name should be generic and search-friendly (e.g., "Apple" not "Small sour green apple")
- Minor variations are acceptable ("Chicken" vs "Chicken Breast" - both valid)
- The name must still refer to the same food as the original Hebrew text
- Compound dishes should be decomposed (Hebrew for "pasta with cheese" -> "Pasta" and "Cheese" separately)
- Individual ingredients from decomposed dishes are valid on their own
- Common food product names are acceptable even if multi-word ("Protein Shake", "Peanut Butter")
- Do NOT penalize names that are already standard food category names
- The name MUST be in English, not Hebrew

Grade as acceptable if a nutrition database search for this name would reasonably find the correct food."""

judge_llm = init_chat_model("gpt-4o", temperature=0).with_structured_output(
    NameGrade, method="json_schema"
)


async def food_name_quality(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """LLM-as-judge: evaluate if food names are reasonable normalizations.

    Skips if no items expected (non-food actions). Returns score 0.0-1.0
    (fraction of names graded acceptable).
    """
    expected_items = reference_outputs.get("items", [])
    actual_items = outputs.get("items", [])

    if not expected_items:
        return {"key": "food_name_quality", "score": 1.0, "comment": "No items to check"}

    if not actual_items:
        return {"key": "food_name_quality", "score": 0.0, "comment": "No items produced"}

    acceptable_count = 0
    details = []

    for act in actual_items:
        msg = (
            f'Original user input (Hebrew): "{inputs["question"]}"\n'
            f'Extracted food name: "{act.get("food_name", "")}"\n'
            f'\nIs this a reasonable, search-friendly English normalization of the Hebrew input?'
        )

        grade = await judge_llm.ainvoke([
            {"role": "system", "content": judge_instructions},
            {"role": "user", "content": msg},
        ])

        if grade["is_acceptable"]:
            acceptable_count += 1
        details.append(
            f"{act.get('food_name', '?')}: "
            f"{'OK' if grade['is_acceptable'] else 'FAIL'} - "
            f"{grade['reasoning'][:80]}"
        )

    score = acceptable_count / len(actual_items)
    return {"key": "food_name_quality", "score": score, "comment": "; ".join(details)}

## Run Evaluation

Execute all evaluators against the dataset and display results.

In [23]:
experiment_results = await client.aevaluate(
    run_input_parser,
    data=dataset_id,
    evaluators=[
        correct_action,
        correct_item_count,
        amount_accuracy,
        correct_dates,
        food_name_quality,
    ],
    experiment_prefix="input-parser-hebrew-gpt4.1-nano",
    max_concurrency=4,
)
experiment_results.to_pandas()

/Users/dolevsa/Developer/fit_pal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'input-parser-hebrew-gpt4.1-nano-74457c76' at:
https://smith.langchain.com/o/df0d054f-18d0-4ce1-9c1e-f8bdd89430f4/datasets/175cb9ae-e063-466c-a79a-c71db1d94ca2/compare?selectedSessions=03cce08b-b50b-459b-89eb-2292a1c02aad




0it [00:00, ?it/s]

2026-04-14 06:44:14 [info     ] Input parsed                   action=QUERY_DAILY_STATS items=0


1it [00:01,  1.17s/it]

...

2026-04-14 06:44:20 [info     ] Input parsed                   action=LOG_FOOD items=1
2026-04-14 06:44:20 [info     ] Input parsed                   action=LOG_FOOD items=1


15it [00:08,  2.32it/s]

2026-04-14 06:44:21 [info     ] Input parsed                   action=QUERY_DAILY_STATS items=0
2026-04-14 06:44:22 [info     ] Input parsed                   action=LOG_FOOD items=1


17it [00:11,  1.54it/s]


,inputs.question,outputs.action,outputs.items,outputs.item_count,outputs.consumed_at,outputs.start_date,outputs.end_date,error,reference.items,reference.action,...,reference.start_date,reference.consumed_at,feedback.correct_action,feedback.correct_item_count,feedback.amount_accuracy,feedback.correct_dates,feedback.food_name_quality,execution_time,example_id,id
0,כמה גרם חלבון אכלתי בממוצע בשבוע האחרון,QUERY_DAILY_STATS,[],0,NaN,2026-04-07,2026-04-13,None,[],QUERY_DAILY_STATS,...,RELATIVE_7_DAYS_AGO,NaN,True,True,1.0,True,1.0,1.159081,0ae465e4-e4fe-4e9e-929f-764bf80e9142,019d8a16-f22c-7bc1-932a-cce2ff164e64
1,מה אכלתי בשבוע האחרון,QUERY_DAILY_STATS,[],0,NaN,2026-04-07,2026-04-14,None,[],QUERY_DAILY_STATS,...,RELATIVE_7_DAYS_AGO,NaN,True,True,1.0,False,1.0,1.814669,2096343b-3132-412d-b597-afde4d496e7f,019d8a16-f6b9-7470-98c6-ffd26bf75df8
2,קפה,LOG_FOOD,"[{'food_name': 'Coffee', 'amount': 240.0, 'uni...",1,NaN,NaN,NaN,None,"[{'amount': 240.0, 'food_name': 'Coffee'}]",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,1.964982,13be27ec-79af-4448-9fb2-04637f9b60ba,019d8a16-f22e-7400-87b8-37db9d31263a
3,אכלתי כוס אורז,LOG_FOOD,"[{'food_name': 'Rice', 'amount': 158.0, 'unit'...",1,NaN,NaN,NaN,None,"[{'amount': 158.0, 'food_name': 'Rice'}]",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,1.957394,08508e44-6d96-4318-9c54-a62bf65a2291,019d8a16-f225-7583-9fcf-65fbafbbb5e6
4,היי מה נשמע?,CHITCHAT,[],0,NaN,NaN,NaN,None,[],CHITCHAT,...,NaN,NaN,True,True,1.0,True,1.0,0.770685,43bfb009-e4d3-475a-9089-35e1592239cc,019d8a16-fe66-7210-9606-cced93d32540
5,כמה חלבון יש בביצה?,QUERY_FOOD_INFO,[],0,NaN,NaN,NaN,None,[],QUERY_FOOD_INFO,...,NaN,NaN,True,True,1.0,True,1.0,0.614441,441c4502-eeb3-4012-92cd-fbe32cdbc3e7,019d8a16-ff57-79f1-8a67-207314771a51
6,אכלתי בננה אתמול,LOG_FOOD,"[{'food_name': 'Banana', 'amount': 118.0, 'uni...",1,2026-04-13 12:00:00+00:00,NaN,NaN,None,"[{'amount': 120.0, 'food_name': 'Banana'}]",LOG_FOOD,...,NaN,YESTERDAY_NOON,True,True,1.0,True,1.0,0.765018,32af0f49-ed96-4127-87b1-96b5008884b2,019d8a16-fdd4-7b50-8128-34cebbe2cba3
7,מה אכלתי היום?,QUERY_DAILY_STATS,[],0,NaN,NaN,NaN,None,[],QUERY_DAILY_STATS,...,NaN,NaN,True,True,1.0,True,1.0,0.569054,54a77370-bb1f-485d-83ae-5f6dfb6a9da3,019d8a17-050f-7d83-a26e-a8f3fa42762c
8,תרשום בננה ו-100 גרם אורז,LOG_FOOD,"[{'food_name': 'Banana', 'amount': 100.0, 'uni...",2,NaN,NaN,NaN,None,"[{'amount': 120.0, 'food_name': 'Banana'}, {'a...",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,1.637036,02c9eb9a-f3c8-4e25-8783-eaded483e99b,019d8a16-f224-7141-aee6-f6bc9913350c
9,2 פרוסות לחם,LOG_FOOD,"[{'food_name': 'Bread', 'amount': 60.0, 'unit'...",1,NaN,NaN,NaN,None,"[{'amount': 60.0, 'food_name': 'Bread'}]",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,0.978208,4ea140bc-ae5e-4efb-8ccf-0d9eb545f836,019d8a17-016d-75b1-a834-aa170f84489e


## Notes

- Re-run this notebook after changing the input parser prompt or switching LLM models
- Compare experiments in LangSmith UI: Datasets & Testing -> Input Parser Hebrew
- To add more examples, update the `examples` list and delete/recreate the dataset
- Amount tolerance is +/-20% — adjust in `amount_accuracy` if too strict/lenient
- Food names should be normalized to **English** even though input is Hebrew